![ATARRI logos](img/logos.png)

# 3.2 MPLNET lidar vs MONARCH dust forecast comparison: Processing forecasts

#### Objective

The objective of this notebook is to process MONARCH dust model data over the Barcelona station corresponding to the dates of the previously analysed dust event, so that it may be compared with MPLNET data.

## MONARCH extinction

Now, we will obtain the same dust extinction profile for our MONARCH datasets at the Barcelona station. Note that, in the [dust.aemet](https://dust.aemet.es/) products, we already have dust extinction. However, this product corresponds only to the surface level, and since we are interested in the entire vertical column, we will apply the following workflow:

- Extract species values from `od550_dust`, `dust_load` and `concdu`.
- Read the intermediate csv files and filter data.
- Calculate the dust extinction at each vertical level.
- Save the final output to a csv file.


## Importing libraries

In [10]:
import xarray as xr
import numpy as np
from pathlib import Path
import netCDF4 as nc
from netCDF4 import num2date
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

In [11]:
from IPython.utils.io import capture_output
with capture_output():
    %run ../functions/VT3-functions.ipynb

For this subsection, we will only need to call the `VT3-functions.ipynb` notebook. We will also need to reset our site coordinates, our data subdirectories and our output directory.

In [12]:
#%% SITE COORDINATES 
sites = {
    "Barcelona": {"latitude": 41.3860, "longitude": 2.1170}
    # you can add as many site coordinates as you wish in this list
}

In [13]:
directory_path_loaddu = Path("/shared/data/exp_to_interp/monarch/regional/3hourly/dust_load")
directory_path_od550du = Path("/shared/data/exp_to_interp/monarch/regional/3hourly/od550_dust")
directory_path_concdu = Path("/shared/data/exp_to_interp/monarch/regional/3hourly/sconc_dust")

In [14]:
output_base_path = Path("../csv")

In [15]:
start_date = datetime(2025, 3, 3)
end_date = datetime(2025, 3, 8)

## Pre-processing

To facilitate our MONARCH analysis, we will make use of the `process_monarch_variable()` function. This function will process each of the MONARCH datasets and save numerical data to csv files. In the case of `sconc_dust`, the output csv will include values at each altitude level. Each .csv is saved directly to our `/csv` folder.

In [16]:
# For dust concentration:
process_monarch_variable("sconc_dust", sites, directory_path_concdu, start_date, end_date, output_base_path)

# For dust load:
process_monarch_variable("dust_load", sites, directory_path_loaddu, start_date, end_date, output_base_path)

# For dust optical depth:
process_monarch_variable("od550_dust", sites, directory_path_od550du, start_date, end_date, output_base_path)



Processing site: Barcelona
Processing file: sconc_dust_2025030312.nc
Processing file: sconc_dust_2025030412.nc
Processing file: sconc_dust_2025030512.nc
Processing file: sconc_dust_2025030612.nc
CSV file saved to ../csv/sconc_dust_Barcelona.csv

Processing site: Barcelona
Processing file: dust_load_2025030312.nc
Processing file: dust_load_2025030412.nc
Processing file: dust_load_2025030512.nc
Processing file: dust_load_2025030612.nc
CSV file saved to ../csv/dust_load_Barcelona.csv

Processing site: Barcelona
Processing file: od550_dust_2025030312.nc
Processing file: od550_dust_2025030412.nc
Processing file: od550_dust_2025030512.nc
Processing file: od550_dust_2025030612.nc
CSV file saved to ../csv/od550_dust_Barcelona.csv


## Reading .csv files

Once the preprocessing is done, we will re-read the output csv files as data frames.

In [17]:
concdu_path = '../csv/sconc_dust_Barcelona.csv'
loaddu_path = '../csv/dust_load_Barcelona.csv'
od550du_path = '../csv/od550_dust_Barcelona.csv'

dust_conc_data = pd.read_csv(concdu_path)
loaddu_data = pd.read_csv(loaddu_path)
od550du_data = pd.read_csv(od550du_path)

If you look closely, you will notice that we again have two sets of data points at midnight.

In [18]:
dust_conc_data

,time,0.00,250.00,500.00,750.00,1000.00,1500.00,2000.00,3000.00,4000.00,5000.00,6000.00,8000.00,10000.00,12000.00
0,2025-03-04 00:00:00,2.387649e-09,3.095478e-09,1.452630e-08,3.144622e-08,3.078621e-08,4.555418e-08,4.318344e-08,6.400119e-08,3.312116e-09,1.410673e-09,1.430221e-09,6.434805e-11,1.241225e-13,1.009119e-19
1,2025-03-04 03:00:00,2.685390e-09,3.792238e-09,1.497859e-08,2.248090e-08,2.538781e-08,7.688739e-08,1.532948e-07,4.407160e-08,2.579770e-09,1.978591e-09,1.368650e-09,5.639573e-11,5.747515e-15,1.682543e-20
2,2025-03-04 06:00:00,4.173618e-09,5.473769e-09,1.482429e-08,3.493940e-08,5.151354e-08,8.736882e-08,9.273366e-08,1.988867e-08,4.662231e-09,1.845425e-09,9.794094e-10,4.411150e-11,1.481190e-13,4.326643e-20
3,2025-03-04 09:00:00,5.575906e-09,6.261387e-09,1.741249e-08,2.331086e-08,3.423384e-08,1.150821e-07,1.120279e-07,2.031249e-08,4.054840e-09,1.987563e-09,1.045998e-09,2.972990e-10,2.684365e-15,9.641685e-20
4,2025-03-04 12:00:00,4.298911e-09,4.780516e-09,1.695290e-08,3.162640e-08,3.322675e-08,7.609557e-08,7.620392e-08,6.337730e-08,3.950463e-08,1.923695e-09,9.403035e-10,2.730575e-10,1.664026e-15,6.627699e-19
5,2025-03-04 15:00:00,5.418448e-09,5.824409e-09,1.085326e-08,4.015672e-08,8.601853e-08,1.029196e-07,6.345603e-08,6.247596e-08,2.824519e-08,2.648393e-09,1.409497e-09,5.640848e-11,3.502266e-16,1.436675e-20
6,2025-03-04 18:00:00,8.676950e-10,7.058441e-10,4.627609e-09,7.054384e-08,1.366671e-07,3.032974e-08,3.988706e-09,2.041166e-09,6.310498e-09,6.006987e-09,4.982395e-09,4.703752e-11,1.167028e-17,3.144440e-21
7,2025-03-04 21:00:00,4.736556e-10,1.669133e-10,4.378369e-08,1.964156e-08,7.985785e-09,9.621727e-08,1.876273e-10,1.504522e-09,3.114932e-09,4.424830e-09,5.846431e-09,7.478172e-12,9.446154e-17,2.415674e-18
8,2025-03-05 00:00:00,8.814809e-09,1.166479e-08,4.088628e-08,1.050794e-07,1.283638e-07,1.347055e-07,9.043258e-08,2.042491e-08,4.464122e-09,1.125495e-09,1.545538e-09,3.578114e-12,3.833440e-15,7.470083e-18
9,2025-03-05 03:00:00,8.879549e-09,9.239462e-09,1.909430e-08,6.672966e-08,1.057941e-07,1.140676e-07,8.072585e-08,2.266081e-08,6.124726e-09,1.322842e-09,2.554032e-09,8.631143e-13,3.410306e-16,3.809472e-18


To get around this, we need to filter the data frames again, taking both values at `00:00:00` and averaging them to produce a unique value for midnight.

In [19]:
#%% REMOVE DUPLICATES 
dust_conc_cleaned = dust_conc_data.groupby('time', as_index=False).mean()
od550du_cleaned = od550du_data.groupby('time', as_index=False).mean()
loaddu_cleaned = loaddu_data.groupby('time', as_index=False).mean()
dust_conc_cleaned

,time,0.00,250.00,500.00,750.00,1000.00,1500.00,2000.00,3000.00,4000.00,5000.00,6000.00,8000.00,10000.00,12000.00
0,2025-03-04 00:00:00,2.387649e-09,3.095478e-09,1.452630e-08,3.144622e-08,3.078621e-08,4.555418e-08,4.318344e-08,6.400119e-08,3.312116e-09,1.410673e-09,1.430221e-09,6.434805e-11,1.241225e-13,1.009119e-19
1,2025-03-04 03:00:00,2.685390e-09,3.792238e-09,1.497859e-08,2.248090e-08,2.538781e-08,7.688739e-08,1.532948e-07,4.407160e-08,2.579770e-09,1.978591e-09,1.368650e-09,5.639573e-11,5.747515e-15,1.682543e-20
2,2025-03-04 06:00:00,4.173618e-09,5.473769e-09,1.482429e-08,3.493940e-08,5.151354e-08,8.736882e-08,9.273366e-08,1.988867e-08,4.662231e-09,1.845425e-09,9.794094e-10,4.411150e-11,1.481190e-13,4.326643e-20
3,2025-03-04 09:00:00,5.575906e-09,6.261387e-09,1.741249e-08,2.331086e-08,3.423384e-08,1.150821e-07,1.120279e-07,2.031249e-08,4.054840e-09,1.987563e-09,1.045998e-09,2.972990e-10,2.684365e-15,9.641685e-20
4,2025-03-04 12:00:00,4.298911e-09,4.780516e-09,1.695290e-08,3.162640e-08,3.322675e-08,7.609557e-08,7.620392e-08,6.337730e-08,3.950463e-08,1.923695e-09,9.403035e-10,2.730575e-10,1.664026e-15,6.627699e-19
5,2025-03-04 15:00:00,5.418448e-09,5.824409e-09,1.085326e-08,4.015672e-08,8.601853e-08,1.029196e-07,6.345603e-08,6.247596e-08,2.824519e-08,2.648393e-09,1.409497e-09,5.640848e-11,3.502266e-16,1.436675e-20
6,2025-03-04 18:00:00,8.676950e-10,7.058441e-10,4.627609e-09,7.054384e-08,1.366671e-07,3.032974e-08,3.988706e-09,2.041166e-09,6.310498e-09,6.006987e-09,4.982395e-09,4.703752e-11,1.167028e-17,3.144440e-21
7,2025-03-04 21:00:00,4.736556e-10,1.669133e-10,4.378369e-08,1.964156e-08,7.985785e-09,9.621727e-08,1.876273e-10,1.504522e-09,3.114932e-09,4.424830e-09,5.846431e-09,7.478172e-12,9.446154e-17,2.415674e-18
8,2025-03-05 00:00:00,8.814809e-09,1.166479e-08,4.088628e-08,1.050794e-07,1.283638e-07,1.347055e-07,9.043258e-08,2.042491e-08,4.464122e-09,1.125495e-09,1.545538e-09,3.578114e-12,3.833440e-15,7.470083e-18
9,2025-03-05 03:00:00,8.879549e-09,9.239462e-09,1.909430e-08,6.672966e-08,1.057941e-07,1.140676e-07,8.072585e-08,2.266081e-08,6.124726e-09,1.322842e-09,2.554032e-09,8.631143e-13,3.410306e-16,3.809472e-18


Next, we need to collect our time and altitude data in variables.

In [20]:
time_dust_conc = dust_conc_data['time']
time_loaddu = loaddu_data['time']
time_od550du = od550du_data['time']

In [21]:
altitudes = [float(col) for col in dust_conc_cleaned.columns[1:]]
altitudes

[0.0,
 250.0,
 500.0,
 750.0,
 1000.0,
 1500.0,
 2000.0,
 3000.0,
 4000.0,
 5000.0,
 6000.0,
 8000.0,
 10000.0,
 12000.0]

As we are going to calculate dust extinction, we need to ensure we match all variables to the same time step.

In [22]:
common_times = set(dust_conc_cleaned['time']).intersection(
    od550du_cleaned['time']
).intersection(
    loaddu_cleaned['time']
)

In [23]:
# Filter each dataset to include only rows with common times
dust_conc_aligned = dust_conc_cleaned[dust_conc_cleaned['time'].isin(common_times)]
od550du_aligned = od550du_cleaned[od550du_cleaned['time'].isin(common_times)]
loaddu_aligned = loaddu_cleaned[loaddu_cleaned['time'].isin(common_times)]

time_dust_conc_aligned = dust_conc_aligned['time']
time_dust_conc_aligned = pd.to_datetime(time_dust_conc_aligned)
dust_conc = dust_conc_aligned.drop(columns=['time'])

time_od550du_aligned = od550du_aligned['time']
time_od550du_aligned = pd.to_datetime(time_od550du_aligned)
od550du = od550du_aligned.drop(columns=['time'])

time_loaddu_aligned = loaddu_aligned['time']
time_loaddu_aligned = pd.to_datetime(time_loaddu_aligned)
loaddu = loaddu_aligned.drop(columns=['time'])

## Calculating dust extinction and saving output

Dust extinction can be calculated with the following formula:

$$
\text{dust extinction} = \left(\frac{\text{dust optical depth at 550nm}}{\text{dust load}}\right) \cdot \text{dust concentration}
$$


Since our `concdu` variable has different altitude levels, we will obtain an extinction result per each level. For the calculation to work, we need to transform `concdu` and the `od550_dust / dust_load` ratio to arrays.

In [24]:
ratio = (od550du / loaddu)
ratio = np.array(ratio)
concdu = np.array(dust_conc)
dust_ext_monarch = ratio * concdu
dust_ext_monarch

array([[1.22883886e-06, 1.59313352e-06, 7.47617505e-06, 1.61842620e-05,
        1.58445802e-05, 2.34451307e-05, 2.22249979e-05, 3.29391560e-05,
        1.70462952e-06, 7.26023818e-07, 7.36084279e-07, 3.31176778e-08,
        6.38814951e-11, 5.19358269e-17],
       [1.30416063e-06, 1.84170177e-06, 7.27435800e-06, 1.09178585e-05,
        1.23296005e-05, 3.73403930e-05, 7.44476641e-05, 2.14033900e-05,
        1.25286657e-06, 9.60903787e-07, 6.64685298e-07, 2.73886108e-08,
        2.79128325e-12, 8.17127807e-18],
       [2.20915767e-06, 2.89734681e-06, 7.84671463e-06, 1.84939414e-05,
        2.72668785e-05, 4.62456040e-05, 4.90852949e-05, 1.05273661e-05,
        2.46778736e-06, 9.76810383e-07, 5.18415829e-07, 2.33488649e-08,
        7.84015907e-11, 2.29015617e-17],
       [2.94736474e-06, 3.30970323e-06, 9.20405877e-06, 1.23218727e-05,
        1.80956472e-05, 6.08312060e-05, 5.92167485e-05, 1.07369665e-05,
        2.14334558e-06, 1.05060501e-06, 5.52903333e-07, 1.57149110e-07,
        1.418

Finally, we can save our resulting extinction (in /m units), together with the MONARCH altitudes, in a csv file with the same `save_to_csv()` function we used previously. We will use this output in the following notebook.

In [25]:
csv_name = "MONARCH_Barcelona_20250304_20250308"

save_to_csv(time_dust_conc_aligned, dust_ext_monarch, csv_name, output_base_path, altitudes)

CSV file saved to ../csv/MONARCH_Barcelona_20250304_20250308.csv


## References and further reading

The analysis, main script and auxiliary functions of this notebook have been adapted from their original version, provided by Carlotta Gilè (BSC-CNS). 

<table style="width:100%;">
    <tr>
        <td style="text-align: center;"><a href="VT3-1-MPLNET_extinction.ipynb" style="font-size: 18px;">⬅ Previous</a></td>
        <td style="text-align: center;"><a href="VT3-3-MPLNET_MONARCH_daily_avg.ipynb" style="font-size: 18px;">Next ➡</a></td>
    </tr
</table>